In [ ]:
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.9/343.9 kB 28.8 MB/s eta 0:00:00


In [ ]:
pip install sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 94.9 MB/s eta 0:00:00


In [ ]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
from datasets import load_dataset

device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model_id = "openai/whisper-large-v3"

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
)
model.to(device)

processor = AutoProcessor.from_pretrained(model_id)

pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    torch_dtype=torch_dtype,
    device=device,
)


config.json:   0%|          | 0.00/1.27k [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.90k [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/494k [00:00<?, ?B/s]

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.07k [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


In [ ]:
dataset = load_dataset("distil-whisper/librispeech_long", "clean", split="validation")
sample = dataset[0]["audio"]

result = pipe(sample, return_timestamps=True)

README.md:   0%|          | 0.00/480 [00:00<?, ?B/s]

clean/validation-00000-of-00001-91350812(…):   0%|          | 0.00/1.98M [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/1 [00:00<?, ? examples/s]

Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits pr

In [ ]:
print(result["text"])

 Mr. Quilter is the apostle of the middle classes, and we are glad to welcome his gospel. Nor is Mr. Quilter's manner less interesting than his matter. He tells us that at this festive season of the year, with Christmas and roast beef looming before us, similes drawn from eating and its results occur most readily to the mind. He has grave doubts whether Sir Frederick Leighton's work is really Greek after all, One can discover in it but little of rocky Ithaca. Linnell's pictures are a sort of Upgards and Adam paintings, and Mason's exquisite idylls are as national as a jingo poem. Mr. Burkett Foster's landscapes smile at one much in the same way that Mr. Carker used to flash his teeth, and Mr. John Collier gives his sitter a cheerful slap on the back before he says, like a shampooer in a Turkish bath. Next man!


In [ ]:
from transformers import pipeline
classifier = pipeline(
    task="text-classification",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=0
)

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

In [ ]:
feedback_result = classifier(result["text"])
print(feedback_result)

[{'label': 'POSITIVE', 'score': 0.9616144299507141}]


In [ ]:
GEMINI_API_KEY = ""

In [ ]:
import google.generativeai as genai

genai.configure(api_key = GEMINI_API_KEY)

model = genai.GenerativeModel("gemini-2.5-flash")
text = result["text"]
prompt = f"""
classify the following text as:
  - spam
  - not spam
  return only the two labels
  text :
  {text}
"""
response = model.generate_content(prompt)

print(response.text)

ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1490.74ms


not spam


In [ ]:
from pypdf import PdfReader

reader = PdfReader("/content/10_chapter 2.pdf")
number_of_pages = len(reader.pages)

In [ ]:
combined_text = ""

In [ ]:
for i in range(number_of_pages):
  page = reader.pages[i]
  text = page.extract_text()
  if text:
    combined_text += text + "\n"

In [ ]:
def chunk(text, chunk_size, overlap):
  words = text.split()
  chunks = []
  start = 0
  while start < len(words):
    end = start + chunk_size
    chunks.append(" ".join(words[start:end]))
    start += chunk_size - overlap
  return chunks

In [ ]:
content = chunk(combined_text, 200, 50)

In [ ]:
embedding.dtype

dtype('float32')

In [ ]:
type(embedding)

numpy.ndarray

In [ ]:
from sentence_transformers import SentenceTransformer
embed = SentenceTransformer("all-MiniLM-L6-v2")
embedding = embed.encode(content)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
dimension = embedding.shape[1]
dimension

384

In [ ]:
import faiss
import numpy as np
index = faiss.IndexFlatL2(dimension)

In [ ]:
index.add(embedding)

In [ ]:
query = "spam message phishing scam advertisement"

query_embedding = embed.encode([query])

In [ ]:
query_embedding = np.array(query_embedding).astype("float32")

distances, indices = index.search(query_embedding, 5)

In [ ]:
print(distances, indices)

[[1.6216434 1.6622032 1.6639146 1.6789038 1.7006614]] [[142  36 100  72   2]]


In [ ]:
retrieved_chunks = [
    content[i]
    for i in indices[0]
]

In [ ]:
rag_context = "\n\n".join(retrieved_chunks)
rag_context

'service. Therefore, employees have an anchor role to play in enhancing the business potentialities of a bank. Marketing products calls for better employee awareness regarding the features of various products offered by a bank. If employees are well informed in this area, they can convince the clientele more easily. Infact this is an area wherein human resources have an edge over machines. A computer may ensure speedier service but it cannot resort to personal selling. So if employees make sincere attempts to seize business on a large scale, there will be substantial increase in quantum of business and thus in per employee business too. It is a well accepted fact that Indian private sector banks have a devoted and well motivated work force. This is one of the reasons for their success in all banking endeavours. With little or no interference in recruitment process, they go in for their own way of selecting people to suit into the business environment. At the same time, they ever fail t

In [ ]:
import google.generativeai as genai

genai.configure(api_key = GEMINI_API_KEY)

model = genai.GenerativeModel("gemini-2.5-flash")
text = result["text"]
prompt = f"""
classify the following text as:
  - spam
  - not spam
  return only the two labels
  text :
  {rag_context}
"""
response = model.generate_content(prompt)

print(response.text)

ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1611.62ms


not spam
